# Motor de Busqueda de Imagenes Similares con DeepLearning

**DIEGO BESADA RODRÍGUEZ**

<div>
  <img src="images/flores.png"
       alt="Flores image"
       width="400">
</div>

<br>

<div style="border:2px solid orange; padding:10px; border-radius:5px; background-color:#fff3cd;">
<strong>Estado:</strong> En desarrollo (Parte 2 - Fine-tuning).
</div>

En esta segunda parte se ajusta el modelo ResNet18 pre-entrenado mediante fine-tuning sobre el dataset de flores objetivo. El objetivo es mejorar la capacidad de extracción de características específicas del dominio.

**Estructura de trabajo de esta parte:**
1. Carga del modelo pre-entrenado desde la Parte 1.
2. Preparación del dataset con augmentación de datos.
3. Fine-tuning de los últimos bloques (finetuning parcial).
4. Evaluación y comparación con el modelo pre-entrenado.
5. Análisis de mejoras en el buscador.

In [ ]:
import json
import warnings
from pathlib import Path
from copy import deepcopy

In [ ]:
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
import seaborn as sns

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim import lr_scheduler
from torch.utils.data import DataLoader, Dataset, random_split
from torchvision import datasets, transforms
import torchvision.models as models

In [ ]:
plt.style.use("default")
plt.rcdefaults()
warnings.filterwarnings("ignore")

main_datasets_path = Path("Datasets/Datasets_image/")

if torch.cuda.is_available():
    device = torch.device("cuda")
    print("✓ Using NVIDIA CUDA GPU")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    print("✓ Using Apple Metal Performance Shaders (MPS)")
else:
    device = torch.device("cpu")
    print("⚠ Using CPU (slower)")

## Carga del modelo y preparación de datos

## Cargar modelo pre-entrenado de la Parte 1

En esta sección cargaremos el modelo ResNet18 pre-entrenado que fue desarrollado en la Parte 1. Este modelo ya ha sido ajustado en el dataset de 102 flores y servirá como punto de partida para el fine-tuning. Utilizaremos este modelo base para mejorar aún más su capacidad de extracción de características mediante el reentrenamiento selectivo de sus últimas capas en el dominio específico de flores.

In [ ]:
pretrained_model_path = "data/model_resnet18_102flowers.pt"

model = models.resnet18(weights=None)
num_features = model.fc.in_features
model.fc = nn.Sequential(
    nn.Linear(num_features, 512),
    nn.ReLU(),
    nn.BatchNorm1d(512),
    nn.Dropout(0.4),
    nn.Linear(512, 102),
    nn.LogSoftmax(dim=1)
)

model.load_state_dict(torch.load(pretrained_model_path, map_location=torch.device('cpu'), weights_only=False)["state_dict"])
model = model.to(device)
model.eval()

### Transformaciones de datos

In [ ]:
# Transformaciones para entrenamiento y validación
train_transforms = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Cargar dataset
dataset_path = main_datasets_path / "Dataset_Flores"
full_dataset = datasets.ImageFolder(str(dataset_path), transform=train_transforms)

# División train-valid (80-20)
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

# Modificar transformaciones de validación
val_dataset.dataset.transform = val_transforms

batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0)

print(f"✓ Dataset cargado: {len(train_dataset)} train, {len(val_dataset)} validación")

## Fine-tuning parcial

Se congela la mayor parte del modelo y se reentrenan las últimas capas convolucionales y el clasificador.

In [ ]:
# Congelar todas las capas excepto layer4 y fc
for name, param in model.named_parameters():
    if 'layer4' not in name and 'fc' not in name:
        param.requires_grad = False

# Contar parámetros entrenables
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())

print(f"Parámetros entrenables: {trainable:,.0f} / {total:,.0f} ({100*trainable/total:.1f}%)")

### Configuración y funciones de entrenamiento

In [ ]:
# Configuración de entrenamiento
num_epochs = 25
learning_rate = 1e-3
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam([p for p in model.parameters() if p.requires_grad], lr=learning_rate)
scheduler = lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3, verbose=True)

# Variables para tracking
history = {
    'train_loss': [],
    'val_loss': [],
    'val_accuracy': []
}

### Ejecución del entrenamiento

In [ ]:
def train_epoch(model, train_loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0
    
    for images, labels in tqdm(train_loader, desc="Training"):
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    
    return total_loss / len(train_loader), 100 * correct / total


def validate_epoch(model, val_loader, criterion, device):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc="Validating"):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            total_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    return total_loss / len(val_loader), 100 * correct / total

In [ ]:
best_val_loss = float('inf')
best_model_state = None

for epoch in range(num_epochs):
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = validate_epoch(model, val_loader, criterion, device)
    
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_accuracy'].append(val_acc)
    
    scheduler.step(val_loss)
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_state = deepcopy(model.state_dict())
    
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1:2d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%")

# Restaurar mejor modelo
model.load_state_dict(best_model_state)
print("\n✓ Entrenamiento completado")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(history['train_loss'], label='Train Loss', marker='o', markersize=3)
axes[0].plot(history['val_loss'], label='Val Loss', marker='s', markersize=3)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Pérdida durante el entrenamiento')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(history['val_accuracy'], label='Val Accuracy', marker='o', markersize=4, color='green')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Precisión en validación')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Evaluación y extracción de embeddings

In [ ]:
# Guardar modelo fine-tuned con nombre DIFERENTE del pre-entrenado
finetuned_model_path = "data/model_resnet18_finetuned.pt"
pretrained_model_path = "data/model_resnet18_102flowers.pt"

# Verificar que NO estamos sobrescribiendo el modelo original
assert finetuned_model_path != pretrained_model_path, "ERROR: Intentando sobrescribir el modelo original"

torch.save(model.state_dict(), finetuned_model_path)

print(f"✓ Modelo original (Parte 1): {pretrained_model_path} [INTACTO]")
print(f"✓ Modelo fine-tuned (Parte 2): {finetuned_model_path} [GUARDADO]")
print(f"✓ Ambos modelos están separados y no interfieren entre sí")

In [ ]:
# Función para extraer embeddings del modelo fine-tuned
def predict_feature_vector_finetuned(image_path, model, device):
    img = Image.open(image_path).convert('RGB')
    preprocess = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    img_tensor = preprocess(img).unsqueeze(0).to(device)
    
    with torch.no_grad():
        original_fc = model.fc
        try:
            model.fc = torch.nn.Identity()
            output = model(img_tensor)
        finally:
            model.fc = original_fc
    
    return output.cpu().detach().numpy().squeeze().astype(np.float32)

# Construir base de datos de embeddings fine-tuned
base = Path(main_datasets_path) / "Dataset_Flores"
flores = {
    carpeta.name: list(carpeta.glob("*.jpg"))
    for carpeta in base.iterdir()
    if carpeta.is_dir()
}

labels_finetuned = []
paths_finetuned = []
features_finetuned = []

for clase, rutas in flores.items():
    for ruta in tqdm(rutas, desc=f"Extrayendo {clase}"):
        try:
            features = predict_feature_vector_finetuned(str(ruta), model, device)
            labels_finetuned.append(clase)
            paths_finetuned.append(str(ruta))
            features_finetuned.append(features)
        except Exception as e:
            print(f"Error: {ruta} - {e}")

X_finetuned = np.vstack(features_finetuned)

feature_db_finetuned = {
    "labels": np.array(labels_finetuned),
    "paths": np.array(paths_finetuned),
    "features": X_finetuned,
    "model_name": "ResNet18_Finetuned",
    "feature_dimension": X_finetuned.shape[1],
    "num_images": len(labels_finetuned)
}

# Guardar base de datos
finetuned_db_path = Path("data/resnet_finetuned_feature_vector_db.joblib")
finetuned_db_path.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(feature_db_finetuned, finetuned_db_path)
print(f"✓ Base de datos fine-tuned creada: {X_finetuned.shape}")

## Conclusiones

La Parte 2 ha realizado fine-tuning del modelo ResNet18 sobre el dataset de flores, reentrenando las últimas capas convolucionales y el clasificador. Los gráficos de pérdida y precisión muestran el progreso del entrenamiento, y la nueva base de datos de embeddings está lista para evaluación comparativa.

En una evaluación posterior se podría comparar el rendimiento del buscador entre el modelo pre-entrenado (Parte 1) y el modelo fine-tuned (Parte 2) para validar si el ajuste al dominio específico mejora la recuperación de imágenes similares.